# Preliminaries

In [5]:
import numpy as np

import pandas as pd

import seaborn as sns

import matplotlib.pyplot as plt



In [6]:
import glob
import os

In [12]:
import os
import pandas as pd
import glob

def simple_spreadsheet_scan(folder_path):
    """Simple version that just lists files and basic properties"""
    
    extensions = ['*.xlsx', '*.xls', '*.csv', '*.ods']
    files_found = []
    
    for ext in extensions:
        files_found.extend(glob.glob(os.path.join(folder_path, ext)))
    
    print(f"Found {len(files_found)} spreadsheet files:")
    print("-" * 60)
    
    for file_path in files_found:
        file_name = os.path.basename(file_path)
        file_size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        file_ext = os.path.splitext(file_path)[1]
        
        print(f"📊 {file_name}")
        print(f"   Type: {file_ext}")
        print(f"   Size: {file_size:.2f} MB")
        print(f"   Path: {file_path}")
        print()

# Usage
folder_path = "mldataset"
simple_spreadsheet_scan(folder_path)

Found 22 spreadsheet files:
------------------------------------------------------------
📊 adolescent#001.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adolescent#001.csv

📊 adolescent#003.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adolescent#003.csv

📊 adolescent#004.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adolescent#004.csv

📊 adolescent#005.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adolescent#005.csv

📊 adolescent#006.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adolescent#006.csv

📊 adolescent#009.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adolescent#009.csv

📊 adolescent#010.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adolescent#010.csv

📊 adult#001.csv
   Type: .csv
   Size: 0.15 MB
   Path: mldataset\adult#001.csv

📊 adult#002.csv
   Type: .csv
   Size: 0.15 MB
   Path: mldataset\adult#002.csv

📊 adult#003.csv
   Type: .csv
   Size: 0.17 MB
   Path: mldataset\adult#003.csv

📊 adult#004.csv
   Type: .csv
 

In [13]:
import os
import pandas as pd
from openpyxl import load_workbook
import glob
from datetime import datetime

def get_spreadsheet_properties(folder_path):
    """
    Parse a folder and get properties of all spreadsheet files
    Supports: .xlsx, .xls, .csv, .ods
    """
    
    # Supported file extensions
    extensions = ['*.xlsx', '*.xls', '*.csv', '*.ods']
    spreadsheet_files = []
    
    # Find all spreadsheet files in the folder
    for ext in extensions:
        spreadsheet_files.extend(glob.glob(os.path.join(folder_path, ext)))
    
    if not spreadsheet_files:
        print("No spreadsheet files found in the specified folder.")
        return []
    
    spreadsheet_data = []
    
    for file_path in spreadsheet_files:
        try:
            file_info = get_file_properties(file_path)
            spreadsheet_data.append(file_info)
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
    
    return spreadsheet_data

def get_file_properties(file_path):
    """Get properties for a single spreadsheet file"""
    
    file_name = os.path.basename(file_path)
    file_size = os.path.getsize(file_path)
    file_extension = os.path.splitext(file_path)[1].lower()
    modified_time = datetime.fromtimestamp(os.path.getmtime(file_path))
    
    properties = {
        'file_name': file_name,
        'file_path': file_path,
        'file_size_bytes': file_size,
        'file_size_mb': round(file_size / (1024 * 1024), 2),
        'file_extension': file_extension,
        'last_modified': modified_time.strftime('%Y-%m-%d %H:%M:%S'),
        'sheets': [],
        'row_count': 0,
        'column_count': 0
    }
    
    try:
        if file_extension in ['.xlsx', '.xls']:
            properties.update(get_excel_properties(file_path))
        elif file_extension == '.csv':
            properties.update(get_csv_properties(file_path))
        elif file_extension == '.ods':
            properties.update(get_ods_properties(file_path))
    except Exception as e:
        properties['error'] = str(e)
    
    return properties

def get_excel_properties(file_path):
    """Get properties for Excel files"""
    properties = {}
    
    # Using openpyxl for metadata
    workbook = load_workbook(file_path, read_only=True)
    properties['sheets'] = workbook.sheetnames
    properties['sheet_count'] = len(workbook.sheetnames)
    
    # Get dimensions for each sheet
    sheet_info = []
    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
        sheet_info.append({
            'name': sheet_name,
            'rows': sheet.max_row,
            'columns': sheet.max_column
        })
    
    properties['sheet_details'] = sheet_info
    
    # Using pandas for data statistics
    try:
        excel_file = pd.ExcelFile(file_path)
        first_sheet_df = pd.read_excel(file_path, sheet_name=0, nrows=1)
        properties['column_count'] = len(first_sheet_df.columns)
        properties['columns_sample'] = first_sheet_df.columns.tolist()
    except:
        pass
    
    workbook.close()
    return properties

def get_csv_properties(file_path):
    """Get properties for CSV files"""
    properties = {}
    
    try:
        # Read first few rows to get structure
        df_sample = pd.read_csv(file_path, nrows=5)
        properties['row_count'] = sum(1 for line in open(file_path)) - 1  # Subtract header
        properties['column_count'] = len(df_sample.columns)
        properties['columns'] = df_sample.columns.tolist()
        properties['data_types'] = {col: str(dtype) for col, dtype in df_sample.dtypes.items()}
    except Exception as e:
        properties['error'] = f"CSV parsing error: {str(e)}"
    
    return properties

def get_ods_properties(file_path):
    """Get properties for ODS files"""
    properties = {}
    
    try:
        ods_file = pd.ExcelFile(file_path, engine='odf')
        properties['sheets'] = ods_file.sheet_names
        properties['sheet_count'] = len(ods_file.sheet_names)
        
        # Get first sheet info
        first_sheet_df = pd.read_excel(file_path, sheet_name=0, nrows=1, engine='odf')
        properties['column_count'] = len(first_sheet_df.columns)
        properties['columns_sample'] = first_sheet_df.columns.tolist()
    except Exception as e:
        properties['error'] = f"ODS parsing error: {str(e)}"
    
    return properties

def display_spreadsheet_info(spreadsheet_data):
    """Display the spreadsheet information in a readable format"""
    
    print("=" * 80)
    print("SPREADSHEET FOLDER ANALYSIS REPORT")
    print("=" * 80)
    
    for i, file_info in enumerate(spreadsheet_data, 1):
        print(f"\n{i}. FILE: {file_info['file_name']}")
        print(f"   Path: {file_info['file_path']}")
        print(f"   Size: {file_info['file_size_mb']} MB ({file_info['file_size_bytes']} bytes)")
        print(f"   Type: {file_info['file_extension']}")
        print(f"   Last Modified: {file_info['last_modified']}")
        
        if 'sheet_count' in file_info:
            print(f"   Sheets: {file_info['sheet_count']}")
            print(f"   Sheet Names: {', '.join(file_info['sheets'])}")
            
            if 'sheet_details' in file_info:
                print("   Sheet Details:")
                for sheet in file_info['sheet_details']:
                    print(f"     - {sheet['name']}: {sheet['rows']} rows, {sheet['columns']} columns")
        
        if 'column_count' in file_info and file_info['column_count'] > 0:
            print(f"   Columns: {file_info['column_count']}")
            if 'columns_sample' in file_info:
                print(f"   Column Names: {file_info['columns_sample']}")
        
        if 'error' in file_info:
            print(f"   ⚠️  Error: {file_info['error']}")
        
        print("-" * 80)

# Usage example
if __name__ == "__main__":
    # Specify your folder path here
    folder_path = input("Enter the folder path: ").strip()
    
    # Remove quotes if user pasted path with quotes
    folder_path = folder_path.strip('"\'')
    
    if not os.path.exists(folder_path):
        print("Folder path does not exist!")
    else:
        spreadsheet_data = get_spreadsheet_properties(folder_path)
        display_spreadsheet_info(spreadsheet_data)

Enter the folder path:  mldataset


SPREADSHEET FOLDER ANALYSIS REPORT

1. FILE: adolescent#001.csv
   Path: mldataset\adolescent#001.csv
   Size: 0.17 MB (180986 bytes)
   Type: .csv
   Last Modified: 2025-09-05 01:55:56
   Columns: 8
--------------------------------------------------------------------------------

2. FILE: adolescent#003.csv
   Path: mldataset\adolescent#003.csv
   Size: 0.17 MB (179550 bytes)
   Type: .csv
   Last Modified: 2025-09-05 01:55:56
   Columns: 8
--------------------------------------------------------------------------------

3. FILE: adolescent#004.csv
   Path: mldataset\adolescent#004.csv
   Size: 0.17 MB (178638 bytes)
   Type: .csv
   Last Modified: 2025-09-05 01:55:56
   Columns: 8
--------------------------------------------------------------------------------

4. FILE: adolescent#005.csv
   Path: mldataset\adolescent#005.csv
   Size: 0.17 MB (179324 bytes)
   Type: .csv
   Last Modified: 2025-09-05 01:55:56
   Columns: 8
--------------------------------------------------------------

In [14]:
import os
import pandas as pd
from openpyxl import load_workbook
import glob
from datetime import datetime

def parse_spreadsheet_folder(folder_path):
    """
    Parse a folder containing spreadsheets and create DataFrames for CSV files
    Returns: 
    - spreadsheet_properties: List of file properties
    - dataframes_dict: Dictionary of DataFrames (for CSV files)
    """
    
    # Supported file extensions
    extensions = ['*.xlsx', '*.xls', '*.csv', '*.ods']
    spreadsheet_files = []
    
    # Find all spreadsheet files in the folder
    for ext in extensions:
        spreadsheet_files.extend(glob.glob(os.path.join(folder_path, ext)))
    
    if not spreadsheet_files:
        print("No spreadsheet files found in the specified folder.")
        return [], {}
    
    spreadsheet_properties = []
    dataframes_dict = {}
    
    for file_path in spreadsheet_files:
        try:
            # Get file properties
            file_info = get_file_properties(file_path)
            spreadsheet_properties.append(file_info)
            
            # Create DataFrame for CSV files
            if file_info['file_extension'] == '.csv':
                df = create_dataframe_from_csv(file_path)
                if df is not None:
                    # Use filename without extension as key
                    file_key = os.path.splitext(file_info['file_name'])[0]
                    dataframes_dict[file_key] = df
                    file_info['dataframe_created'] = True
                    file_info['dataframe_shape'] = df.shape
                else:
                    file_info['dataframe_created'] = False
            else:
                file_info['dataframe_created'] = "Only CSV files are auto-converted to DataFrames"
                
        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
    
    return spreadsheet_properties, dataframes_dict

def get_file_properties(file_path):
    """Get properties for a single spreadsheet file"""
    
    file_name = os.path.basename(file_path)
    file_size = os.path.getsize(file_path)
    file_extension = os.path.splitext(file_path)[1].lower()
    modified_time = datetime.fromtimestamp(os.path.getmtime(file_path))
    
    properties = {
        'file_name': file_name,
        'file_path': file_path,
        'file_size_bytes': file_size,
        'file_size_mb': round(file_size / (1024 * 1024), 2),
        'file_extension': file_extension,
        'last_modified': modified_time.strftime('%Y-%m-%d %H:%M:%S'),
        'sheets': [],
        'row_count': 0,
        'column_count': 0
    }
    
    try:
        if file_extension in ['.xlsx', '.xls']:
            properties.update(get_excel_properties(file_path))
        elif file_extension == '.csv':
            properties.update(get_csv_properties(file_path))
        elif file_extension == '.ods':
            properties.update(get_ods_properties(file_path))
    except Exception as e:
        properties['error'] = str(e)
    
    return properties

def get_csv_properties(file_path):
    """Get properties for CSV files"""
    properties = {}
    
    try:
        # Read first few rows to get structure
        df_sample = pd.read_csv(file_path, nrows=5)
        properties['row_count'] = sum(1 for line in open(file_path)) - 1  # Subtract header
        properties['column_count'] = len(df_sample.columns)
        properties['columns'] = df_sample.columns.tolist()
        properties['data_types'] = {col: str(dtype) for col, dtype in df_sample.dtypes.items()}
        
        # Additional CSV-specific properties
        with open(file_path, 'r') as f:
            first_line = f.readline()
            properties['delimiter'] = 'comma'  # Default, can be enhanced to detect actual delimiter
            properties['encoding'] = 'utf-8'   # Default, can be enhanced to detect encoding
    except Exception as e:
        properties['error'] = f"CSV parsing error: {str(e)}"
    
    return properties

def create_dataframe_from_csv(file_path, **kwargs):
    """
    Create a DataFrame from a CSV file with error handling
    
    Parameters:
    - file_path: Path to the CSV file
    - **kwargs: Additional pandas read_csv parameters
    
    Returns:
    - DataFrame if successful, None if failed
    """
    try:
        # Try reading with default parameters first
        df = pd.read_csv(file_path, **kwargs)
        print(f"✅ Successfully created DataFrame from {os.path.basename(file_path)}")
        print(f"   Shape: {df.shape}")
        return df
    except Exception as e:
        print(f"❌ Error reading {file_path}: {str(e)}")
        
        # Try alternative encodings if UTF-8 fails
        encodings = ['latin-1', 'iso-8859-1', 'cp1252']
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, encoding=encoding, **kwargs)
                print(f"✅ Successfully read with {encoding} encoding")
                print(f"   Shape: {df.shape}")
                return df
            except:
                continue
        
        print(f"❌ Failed to read {file_path} with all attempted encodings")
        return None

def get_excel_properties(file_path):
    """Get properties for Excel files"""
    properties = {}
    
    workbook = load_workbook(file_path, read_only=True)
    properties['sheets'] = workbook.sheetnames
    properties['sheet_count'] = len(workbook.sheetnames)
    
    sheet_info = []
    for sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
        sheet_info.append({
            'name': sheet_name,
            'rows': sheet.max_row,
            'columns': sheet.max_column
        })
    
    properties['sheet_details'] = sheet_info
    
    try:
        excel_file = pd.ExcelFile(file_path)
        first_sheet_df = pd.read_excel(file_path, sheet_name=0, nrows=1)
        properties['column_count'] = len(first_sheet_df.columns)
        properties['columns_sample'] = first_sheet_df.columns.tolist()
    except:
        pass
    
    workbook.close()
    return properties

def get_ods_properties(file_path):
    """Get properties for ODS files"""
    properties = {}
    
    try:
        ods_file = pd.ExcelFile(file_path, engine='odf')
        properties['sheets'] = ods_file.sheet_names
        properties['sheet_count'] = len(ods_file.sheet_names)
        
        first_sheet_df = pd.read_excel(file_path, sheet_name=0, nrows=1, engine='odf')
        properties['column_count'] = len(first_sheet_df.columns)
        properties['columns_sample'] = first_sheet_df.columns.tolist()
    except Exception as e:
        properties['error'] = f"ODS parsing error: {str(e)}"
    
    return properties

def display_spreadsheet_info(spreadsheet_properties, dataframes_dict):
    """Display the spreadsheet information and DataFrame details"""
    
    print("=" * 80)
    print("SPREADSHEET FOLDER ANALYSIS REPORT")
    print("=" * 80)
    
    for i, file_info in enumerate(spreadsheet_properties, 1):
        print(f"\n{i}. FILE: {file_info['file_name']}")
        print(f"   Path: {file_info['file_path']}")
        print(f"   Size: {file_info['file_size_mb']} MB")
        print(f"   Type: {file_info['file_extension']}")
        print(f"   Last Modified: {file_info['last_modified']}")
        
        if file_info['file_extension'] == '.csv':
            print(f"   Rows: {file_info.get('row_count', 'N/A')}")
            print(f"   Columns: {file_info.get('column_count', 'N/A')}")
            if 'columns' in file_info:
                print(f"   Column Names: {file_info['columns'][:5]}...")  # Show first 5 columns
        
        if 'dataframe_created' in file_info:
            if file_info['dataframe_created'] is True:
                print(f"   ✅ DataFrame Created: Yes")
                print(f"   DataFrame Shape: {file_info.get('dataframe_shape', 'N/A')}")
            else:
                print(f"   ❌ DataFrame Created: No")
        
        if 'error' in file_info:
            print(f"   ⚠️  Error: {file_info['error']}")
        
        print("-" * 80)
    
    # Display DataFrame summary
    if dataframes_dict:
        print(f"\n📊 DATA FRAMES CREATED ({len(dataframes_dict)}):")
        print("=" * 50)
        for df_name, df in dataframes_dict.items():
            print(f"\nDataFrame: '{df_name}'")
            print(f"Shape: {df.shape}")
            print(f"Columns: {list(df.columns)}")
            print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB")
            print(f"First 3 rows:")
            print(df.head(3))
            print("-" * 40)

# Advanced function to create DataFrames from specific sheets in Excel files
def create_dataframes_from_excel(file_path, sheet_names=None):
    """
    Create DataFrames from specific sheets in an Excel file
    
    Parameters:
    - file_path: Path to Excel file
    - sheet_names: List of sheet names to read (if None, reads all sheets)
    
    Returns:
    - Dictionary of DataFrames with sheet names as keys
    """
    try:
        if sheet_names is None:
            # Read all sheets
            dfs = pd.read_excel(file_path, sheet_name=None)
        else:
            # Read specific sheets
            dfs = {}
            for sheet in sheet_names:
                dfs[sheet] = pd.read_excel(file_path, sheet_name=sheet)
        
        print(f"✅ Created {len(dfs)} DataFrames from {os.path.basename(file_path)}")
        return dfs
    except Exception as e:
        print(f"❌ Error reading Excel file {file_path}: {str(e)}")
        return {}

# Usage example
if __name__ == "__main__":
    # Specify your folder path here
    folder_path = input("Enter the folder path: ").strip()
    folder_path = folder_path.strip('"\'')
    
    if not os.path.exists(folder_path):
        print("Folder path does not exist!")
    else:
        # Get properties and create DataFrames for CSV files
        spreadsheet_properties, dataframes_dict = parse_spreadsheet_folder(folder_path)
        
        # Display results
        display_spreadsheet_info(spreadsheet_properties, dataframes_dict)
        
        # Now you can work with the DataFrames
        print(f"\n🎯 ACCESSING YOUR DATA FRAMES:")
        print("=" * 40)
        
        for df_name, df in dataframes_dict.items():
            print(f"\nDataFrame: {df_name}")
            print(f"Available as: dataframes_dict['{df_name}']")
            print(f"Shape: {df.shape}")
            
            # Example: Show basic statistics for numeric columns
            numeric_cols = df.select_dtypes(include=['number']).columns
            if len(numeric_cols) > 0:
                print(f"Numeric columns: {list(numeric_cols)}")
        
        # Example of how to use the DataFrames
        if dataframes_dict:
            print(f"\n💡 EXAMPLE USAGE:")
            first_df_name = list(dataframes_dict.keys())[0]
            first_df = dataframes_dict[first_df_name]
            print(f"# Access first DataFrame: dataframes_dict['{first_df_name}']")
            print(f"# Get shape: dataframes_dict['{first_df_name}'].shape")
            print(f"# Get columns: dataframes_dict['{first_df_name}'].columns.tolist()")
            print(f"# Show first 5 rows: dataframes_dict['{first_df_name}'].head()")

Enter the folder path:  mldataset


✅ Successfully created DataFrame from adolescent#001.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adolescent#003.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adolescent#004.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adolescent#005.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adolescent#006.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adolescent#009.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adolescent#010.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adult#001.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adult#002.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adult#003.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adult#004.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adult#005.csv
   Shape: (1440, 8)
✅ Successfully created DataFrame from adult#006.csv
   Shape: (1440, 8)
✅ Successfully created DataFr

In [ ]:
adolescent#001.csv

In [7]:
# get all spreadsheet files in the folder

folder_path = 'mldataset'
spreadsheet_files = glob.glob(os.path.join(folder_path, '*.csv'))

#Create list of DataFrames
dataframes_list = [pd.read_csv(file) for file in spreadsheet_files]

In [10]:
dataframes_list[0]

,Time,BG,CGM,CHO,insulin,LBGI,HBGI,Risk
0,2023-10-25 06:00:00,126.013943,136.435033,0.0,0.013933,0.0,0.446600,0.446600
1,2023-10-25 06:05:00,126.589661,137.121412,0.0,0.013933,0.0,0.483302,0.483302
2,2023-10-25 06:10:00,127.155902,138.398018,0.0,0.013933,0.0,0.520644,0.520644
3,2023-10-25 06:15:00,127.712577,140.060899,0.0,0.013933,0.0,0.558542,0.558542
4,2023-10-25 06:20:00,128.259611,141.830932,0.0,0.013933,0.0,0.596914,0.596914
...,...,...,...,...,...,...,...,...
1435,2023-10-30 05:35:00,123.071909,119.250427,0.0,0.013933,0.0,0.279643,0.279643
1436,2023-10-30 05:40:00,123.667039,120.113806,0.0,0.013933,0.0,0.310569,0.310569
1437,2023-10-30 05:45:00,124.254709,120.864200,11.6,0.013933,0.0,0.342545,0.342545
1438,2023-10-30 05:50:00,124.834743,121.494518,0.0,0.980600,0.0,0.375486,0.375486


In [2]:
import pandas as pd
import glob

# Path to your folder
#folder_path = ""

# Get list of all CSV files in that folder
csv_files = glob.glob("/*.csv")

# Create a list to store DataFrames
dfs = ['adult#001', 'adult#002',]

# Loop through each file and read into DataFrame
for file in csv_files:
    df = pd.read_csv(file)
    dfs.append(df)

# Now dfs[0], dfs[1], ... contain your DataFrames


NameError: name 'df0' is not defined

In [4]:
teen1 = pd.read_csv("adolescent#001.csv")

teen1

,Time,BG,CGM,CHO,insulin,LBGI,HBGI,Risk
0,2023-10-25 06:00:00,126.013943,136.435033,0.0,0.013933,0.0,0.446600,0.446600
1,2023-10-25 06:05:00,126.589661,137.121412,0.0,0.013933,0.0,0.483302,0.483302
2,2023-10-25 06:10:00,127.155902,138.398018,0.0,0.013933,0.0,0.520644,0.520644
3,2023-10-25 06:15:00,127.712577,140.060899,0.0,0.013933,0.0,0.558542,0.558542
4,2023-10-25 06:20:00,128.259611,141.830932,0.0,0.013933,0.0,0.596914,0.596914
...,...,...,...,...,...,...,...,...
1435,2023-10-30 05:35:00,123.071909,119.250427,0.0,0.013933,0.0,0.279643,0.279643
1436,2023-10-30 05:40:00,123.667039,120.113806,0.0,0.013933,0.0,0.310569,0.310569
1437,2023-10-30 05:45:00,124.254709,120.864200,11.6,0.013933,0.0,0.342545,0.342545
1438,2023-10-30 05:50:00,124.834743,121.494518,0.0,0.980600,0.0,0.375486,0.375486


In [10]:
teen1 = pd.read_csv("ML2324_CW_DataSet_Initial\adolescent#001.csv")
'''
teen3 = pd.read_csv('/content/adolescent#003.csv')

teen4 = pd.read_csv('/content/adolescent#004.csv')

teen5 = pd.read_csv('/content/adolescent#005.csv')

teen6 = pd.read_csv('/content/adolescent#006.csv')

teen9 = pd.read_csv('/content/adolescent#009.csv')

teen10 = pd.read_csv('/content/adolescent#010.csv')

##############################################################

adult1 = pd.read_csv('/content/adult#001.csv')

adult2 = pd.read_csv('/content/adult#002.csv')

adult3 = pd.read_csv('/content/adult#003.csv')

adult4 = pd.read_csv('/content/adult#004.csv')

adult5 = pd.read_csv('/content/adult#005.csv')

adult6 = pd.read_csv('/content/adult#006.csv')

adult7 = pd.read_csv('/content/adult#007.csv')

adult8 = pd.read_csv('/content/adult#008.csv')

adult9 = pd.read_csv('/content/adult#009.csv')

adult10 = pd.read_csv('/content/adult#010.csv')

###################################################################


child1 = pd.read_csv('/content/child#001.csv')

child2 = pd.read_csv('/content/child#002.csv')

child4 = pd.read_csv('/content/child#004.csv')

child5 = pd.read_csv('/content/child#005.csv')

child10 = pd.read_csv('/content/child#010.csv')
'''

OSError: [Errno 22] Invalid argument: 'ML2324_CW_DataSet_Initial\x07dolescent#001.csv'

In [8]:
teen1

NameError: name 'teen1' is not defined

In [ ]:
child10

In [ ]:
# Define all variables in a dictionary with their names as keys
pateints = {
    "child1": child1,
    "child2": child2,
    "child4": child4,
    "child5": child5,
    "child10": child10,
    "teen1": teen1,
    "teen3": teen3,
    "teen4": teen4,
    "teen5": teen5,
    "teen6": teen6,
    "teen9": teen9,
    "teen10": teen10,
    "adult1": adult1,
    "adult2": adult2,
    "adult3": adult3,
    "adult4": adult4,
    "adult5": adult5,
    "adult6": adult6,
    "adult7": adult7,
    "adult8": adult8,
    "adult9": adult9,
    "adult10": adult10,

}

# Iterate through the dictionary and print the names and shapes
for name, df in pateints.items():
    print(f"{name} shape: {df.shape}")


In [ ]:
# Define all variables in a dictionary with their names as keys
pateints_time = {
    "child1": child1['Time'],
    "child2": child2['Time'],
    "child4": child4['Time'],
    "child5": child5['Time'],
    "child10": child10['Time'],
    "teen1": teen1['Time'],
    "teen3": teen3['Time'],
    "teen4": teen4['Time'],
    "teen5": teen5['Time'],
    "teen6": teen6['Time'],
    "teen9": teen9['Time'],
    "teen10": teen10['Time'],
    "adult1": adult1['Time'],
    "adult2": adult2['Time'],
    "adult3": adult3['Time'],
    "adult4": adult4['Time'],
    "adult5": adult5['Time'],
    "adult6": adult6['Time'],
    "adult7": adult7['Time'],
    "adult8": adult8['Time'],
    "adult9": adult9['Time'],
    "adult10": adult10['Time']

}



'''
# Iterate through the dictionary and print the names and shapes
for name, df in pateints.items():
    print(f"{name} shape: {df.shape}")
'''

In [ ]:
time_df = pd.DataFrame.from_dict(pateints_time)

time_df

In [ ]:
time_df['child1']

In [ ]:
# Get unique values row-wise
#time_df["unique_values"] = time_df.apply(lambda row: list(set(row)), axis=1)

time_df["unique_values"]

In [ ]:
# Count unique values in each row
time_df["unique_count"] = time_df.apply(lambda row: len(set(row)), axis=1)

# Filter rows with more than 1 unique value
#rows_with_multiple_unique = time_df[time_df["unique_count"] > 1]


In [ ]:
time_df.dtypes

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Example data: Last 24 hours of CGM and insulin intake data
# This would ideally be real data from a CSV or database
data = {
    'time': pd.date_range(start='2025-08-29', periods=288, freq='5T'),  # 288 samples for 24 hours
    'CGM': np.random.randint(80, 200, 288),  # Random CGM values between 80 and 200
    'insulin': np.random.uniform(1, 10, 288)  # Random insulin intake values between 1 and 10 units
}

df = pd.DataFrame(data)

# For simplicity, assume hypoglycemia occurs if CGM < 70
df['hypoglycemia'] = np.where(df['CGM'] < 70, 1, 0)

# Let's use the last 24 hours of data to predict the next 2 hours
# Extracting features: mean, min, max, std for CGM and insulin over the last 24 hours
def create_features(df, window_size=24):
    features = []
    labels = []

    for i in range(window_size, len(df) - 2):  # The last 2 hours for prediction
        window_cgm = df['CGM'][i-window_size:i]
        window_insulin = df['insulin'][i-window_size:i]

        # Create features: Mean, max, and std for CGM and insulin over the past 24 hours
        mean_cgm = window_cgm.mean()
        max_cgm = window_cgm.max()
        std_cgm = window_cgm.std()
        mean_insulin = window_insulin.mean()
        max_insulin = window_insulin.max()

        features.append([mean_cgm, max_cgm, std_cgm, mean_insulin, max_insulin])

        # The label is whether hypoglycemia will happen in the next 2 hours
        labels.append(df['hypoglycemia'][i+2])  # Prediction for the next 2 hours

    return np.array(features), np.array(labels)

# Create features and labels
X, y = create_features(df)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print(classification_report(y_test, y_pred))

